In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install wandb -qU
!pip install opencv-python kagglehub contractions num2words inflect jieba cn2an sacrebleu safetensors datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.3/118.3 kB 11.6 MB/s 

In [3]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import unicodedata
import contractions
import inflect

import num2words
import cn2an
import jieba

import re
from nltk.corpus import stopwords
from num2words import num2words

import pandas as pd

from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler

from datasets import load_dataset
from transformers import T5ForConditionalGeneration, T5Tokenizer
from safetensors.torch import load_file
from sklearn.model_selection import train_test_split

import wandb
import random

In [5]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

model_weights_path = "drive/MyDrive/T5_fine_tuned_result/model_weights"
tokenizer_path = "drive/MyDrive/T5_fine_tuned_result/tokenizer_weights"

# Загрузка модели
model = T5ForConditionalGeneration.from_pretrained(model_weights_path)
tokenizer = T5Tokenizer.from_pretrained(tokenizer_path)

In [6]:
class CleanChineseText:
    def __init__(self, zh_text):
        self.zh_text = zh_text
    def escape_en_char(self):
        en_pattern = "[a-zA-Z]+"
        self.zh_text = re.sub(en_pattern, ' ', self.zh_text)
        return self
    def remove_enter_and_symbols(self):
        self.zh_text = re.sub(r'^\d+\.', '', self.zh_text).strip()
        return self
    def remove_gaps(self):
        self.zh_text = self.zh_text.replace(' ', '')
        return self
    def remove_punct(self):
        self.zh_text = self.zh_text.strip('，')
        return self

    def clean_all(self):
        # self.remove_punct()
        self.remove_enter_and_symbols()
        self.escape_en_char()
        self.remove_gaps()
        return (self.remove_enter_and_symbols()
          .escape_en_char()
          .remove_gaps()
          .remove_punct()
          .zh_text)

In [7]:
class CleanRussianText:
    def __init__(self, ru_text, remove_punctuation=True, convert_numbers=False):
        self.ru_text = ru_text
        self.remove_punctuation = remove_punctuation
        # self.convert_numbers = convert_numbers

    def to_one_style(self):
      # Lower case
      self.ru_text = self.ru_text.lower().strip()
      return self

    def remove_punctuation_symbols(self):
      if self.remove_punctuation:
        self.ru_text = re.sub(r'[^\w\s]', ' ', self.ru_text)  # пробел вместо пунктуации
        self.ru_text = re.sub(r'\s+', ' ', self.ru_text).strip()
      return self

    def combine_numbers(self):
      # combine numbers with a space, such as '40 000' to '40000'
      self.ru_text = re.sub(r'(?<=\d)\s(?=\d)', '', self.ru_text)
      return self

    def separate_numbers_and_words(self):
      # separate words like '11летнему' to '11 летнему'
      self.ru_text = re.sub(r'(\d+)([а-яА-Я]+)', r'\1 \2', self.ru_text)
      return self

    def separate_numbers_and_words(self):
        # separate words like '11летнему' to '11 летнему'
        self.ru_text = re.sub(r'(\d+)([а-яА-Я]+)', r'\1 \2', self.ru_text)
        return self

    def remove_double_spaces(self):
      self.ru_text = re.sub(r'\s+', ' ', self.ru_text)
      return self

    def remove_duplicate_symbols(self):
      # remove symbols like '????'
      self.ru_text = re.sub(r'(.)\1{3,}', r'\1\1', self.ru_text)
      return self

    def clean_all(self):
      # print(self.ru_text)
      return (self.to_one_style()
              .separate_numbers_and_words()
              .combine_numbers()
              .remove_punctuation_symbols()
              .remove_duplicate_symbols()
              .ru_text)

In [8]:
text = " ♫ \n как лёд сходит, и 3 000 лет идет к 11летнему Тиму ♫ 2025 год ????? 'Телефон 375 44 654 41 51. В городе 40 000 человек.'"
result = CleanRussianText(text).clean_all()
print(result)

как лёд сходит и 3000 лет идет к 11 летнему тиму 2025 год телефон 375446544151 в городе 400 человек


In [9]:
subtitle_dataset = load_dataset("open_subtitles", lang1="ru", lang2="zh_cn", trust_remote_code=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.45k [00:00<?, ?B/s]

open_subtitles.py:   0%|          | 0.00/6.22k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [10]:
dataset_subset = pd.DataFrame(subtitle_dataset['train'])

In [11]:
df = dataset_subset[:700000]

In [12]:
df['cleaned_ru'] = df['translation'].apply(lambda x: CleanRussianText(x['ru']).clean_all())
df['cleaned_zh'] = df['translation'].apply(lambda x: CleanChineseText(x['zh_cn']).clean_all())

<ipython-input-12-41553e1c6f80>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['cleaned_ru'] = df['translation'].apply(lambda x: CleanRussianText(x['ru']).clean_all())
<ipython-input-12-41553e1c6f80>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['cleaned_zh'] = df['translation'].apply(lambda x: CleanChineseText(x['zh_cn']).clean_all())


In [13]:
len(df['cleaned_ru'])

700000

In [14]:
source_ru_train_subt, source_ru_val_subt, target_zh_train_subt, target_zh_val_subt = train_test_split(
    df['cleaned_ru'], df['cleaned_zh'], test_size=0.2, random_state=42)

In [15]:
def tokenize_dataset(source_ru, target_zh, tokenizer=tokenizer):
    prefix = 'translate to zh'
    source_ru_texts = [f'{prefix}: {text}' for text in source_ru]
    tokenized_ru_texts = tokenizer(source_ru_texts, truncation=True, padding=True, return_tensors='pt')
    target_zh_texts = tokenizer(target_zh.tolist(), truncation=True, padding=True, return_tensors='pt')
    return tokenized_ru_texts, target_zh_texts

class TokenizedTedCustomDataset(Dataset):
    def __init__(self, tokenized_ru_texts, target_zh_texts):
        self.ru_input_ids = tokenized_ru_texts['input_ids']
        self.attention_mask = tokenized_ru_texts['attention_mask']

        # self.target_zh = target_zh_texts_ids['input_ids']

        self.labels = target_zh_texts['input_ids']
        self.labels_mask = target_zh_texts['attention_mask']

    # def __getitem__(self, idx):
    #     return {
    #         'input_ids': self.source_ru[idx],
    #         # 'attention_mask': self.source_ru[idx]['attention_mask'].squeeze(0),
    #         'labels': torch.tensor([-100 if x == 0 else x for x in self.target_zh[idx]])
    #     }
    def __getitem__(self, idx):
        labels = self.labels[idx].clone()
        labels_mask = self.labels_mask[idx]

        # маскируем только паддинги
        labels[labels_mask == 0] = -100

        return {
            'input_ids': self.ru_input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': labels
        }

    def __len__(self):
        return len(self.ru_input_ids)


In [16]:
tokenized_ru_texts_train, target_zh_texts_train = tokenize_dataset(source_ru_train_subt, target_zh_train_subt)
tokenized_ru_texts_val, target_zh_texts_val = tokenize_dataset(source_ru_val_subt, target_zh_val_subt)

In [17]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

train_dataset = TokenizedTedCustomDataset(tokenized_ru_texts=tokenized_ru_texts_train, target_zh_texts=target_zh_texts_train)
test_dataset = TokenizedTedCustomDataset(tokenized_ru_texts=tokenized_ru_texts_val, target_zh_texts=target_zh_texts_val)

train_loader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=False, collate_fn=data_collator)
test_loader = DataLoader(dataset=test_dataset, batch_size=256, shuffle=False, collate_fn=data_collator)

In [18]:
from torch.utils.data import Subset

# Generate subset for bleu score
subset_indices = list(range(2000))
bleu_subset = Subset(test_dataset, subset_indices)

# Generate DataLoader for bleu score
bleu_loader = DataLoader(dataset=bleu_subset, batch_size=128, shuffle=False, collate_fn=data_collator)

In [19]:
# from transformers import T5ForConditionalGeneration, T5Tokenizer

# device = 'cuda' #or 'cpu' for translate on cpu

# model_name = 'utrobinmv/t5_translate_en_ru_zh_small_1024'

# model = T5ForConditionalGeneration.from_pretrained(model_name)
# model.to(device)
# tokenizer = T5Tokenizer.from_pretrained(model_name)

# # model = T5ForConditionalGeneration.from_pretrained(pretrained_model_path)
# # model.to(device)
# # tokenizer = T5Tokenizer.from_pretrained(pretrained_tokenizer_path)

# prefix = 'translate to zh: '
# src_text = prefix + "Сдается квартира в хорошем состоянии на год и более."

# # translate Russian to Chinese
# input_ids = tokenizer(src_text, return_tensors="pt")

# generated_tokens = model.generate(**input_ids.to(device))

# result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
# print(result)
# #开发的目的就是向用户提供个性化的同步翻译。


In [20]:
from nltk import bleu
from sacrebleu import corpus_bleu
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

smoothing = SmoothingFunction().method1
import time

def bleu_score(model, tokenizer, eval_pred_loader, device):
    bleu_dic = {}
    actual, predicted = [], []
    model.to(device)
    model.eval()

    with torch.no_grad():
        for batch in eval_pred_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Восстановление паддингов вместо -100
            decoded_labels = labels.clone()
            decoded_labels[decoded_labels == -100] = tokenizer.pad_token_id



            # Генерация предсказаний
            # Когда ты генерируешь outputs = model.generate(input_ids),
            # иногда outputs содержит неправильные ID токенов, которых нет в словаре токенизатора (tokenizer).
            # Это вызывает ошибку в sentencepiece, которая стоит за T5Tokenizer!
            # Иными словами: модель генерит токен id, которого нет в твоём файле токенизатора.
            start = time.time()

            outputs = model.generate(
                input_ids,
                max_length=64,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
)
            print(f"Generate took: {time.time() - start:.2f}s")

            preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            targets = tokenizer.batch_decode(decoded_labels, skip_special_tokens=True)

            # Токенизация ПО СИМВОЛЬНО для китайского языка
            actual.extend([[list(t)] for t in targets])
            predicted.extend([list(p) for p in preds])
            # print(actual, predicted)
    # Оптимизированный подсчет BLEU
    bleu_score = corpus_bleu(actual, predicted)
    bleu_dic['1-2-grams'] = corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing)
    bleu_dic['1-3-grams'] = corpus_bleu(actual, predicted, weights=(0.3, 0.3, 0.3, 0), smoothing_function=smoothing)
    bleu_dic['1-4-grams'] = corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)

    return {"bleu": bleu_score, 'bleu_dic': bleu_dic}

In [21]:
len(bleu_loader.dataset)


2000

In [22]:
bleu_test_subt = bleu_score(model, tokenizer, bleu_loader, device)
print(bleu_test_subt)

/usr/local/lib/python3.11/dist-packages/transformers/data/data_collator.py:741: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
Both `max_new_tokens` (=1024) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generate took: 146.63s


Both `max_new_tokens` (=1024) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generate took: 67.05s


Both `max_new_tokens` (=1024) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Generate took: 1.53s


Both `max_new_tokens` (=1024) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generate took: 66.95s


Both `max_new_tokens` (=1024) and `max_length`(=64) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generate took: 44.55s
{'bleu': 0.009024002737418348, 'bleu_dic': {'1-2-grams': 0.023549877126483366, '1-3-grams': 0.02192079109998201, '1-4-grams': 0.009024002737418348}}


In [23]:
{
  'bleu': 0.1225,   # = 12.25%
  'bleu_dic': {
    '1-2-grams': 0.2282,   # = 22.82%
    '1-3-grams': 0.1983,   # = 19.83%
    '1-4-grams': 0.1225    # = 12.25%
  }
}

{'bleu': 0.1225,
 'bleu_dic': {'1-2-grams': 0.2282, '1-3-grams': 0.1983, '1-4-grams': 0.1225}}

In [24]:
{'bleu': 0.017452313239240094, 'bleu_dic': {'1-2-grams': 0.04230905551591624, '1-3-grams': 0.038562279660033116, '1-4-grams': 0.017452313239240094}}



{'bleu': 0.017452313239240094,
 'bleu_dic': {'1-2-grams': 0.04230905551591624,
  '1-3-grams': 0.038562279660033116,
  '1-4-grams': 0.017452313239240094}}

after tesing:
 {'bleu': 0.01817816826614059, 'bleu_dic': {'1-2-grams': 0.044103026601416365, '1-3-grams': 0.040001565361583495, '1-4-grams': 0.01817816826614059}}


In [ ]:
from transformers import Trainer, TrainingArguments

# Use wandb-core, temporary for wandb's new backend
wandb.require("core")
wandb.login()

training_args = TrainingArguments(
    max_grad_norm=1.0,
    output_dir='/drive/MyDrive/T5_fine_tuned_result/new/',
    per_device_train_batch_size=128,
    per_device_eval_batch_size=256,
    learning_rate=3e-5,
    num_train_epochs=3,
    logging_steps=2000,
    save_steps=5000,
    save_total_limit=1,
    seed=42,
    fp16=False,
    report_to="wandb",
    gradient_accumulation_steps=2
)


total_runs = 5

# This simple block simulates a training loop logging metrics
for run in range(total_runs):
    run_name = f"experiment_{run}"

    # Настраиваем wandb вручную для имени эксперимента
    wandb.init(project="zhrentybot", name=run_name)

    # Создаём тренер
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )

    trainer.train()

    # Имитируем тренировку с логгингом метрик
    epochs = 2
    offset = random.random() / 5
    for epoch in range(2, epochs):
        acc = 1 - 2 ** -epoch - random.random() / epoch - offset
        loss = 2 ** -epoch + random.random() / epoch + offset
        wandb.log({"acc": acc, "loss": loss, "epoch": epoch})

    wandb.finish()

wandb: WARNING `wandb.require('core')` is redundant as it is now the default behavior.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kotishova-lena (kotishova-lena-nonesuch-records) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

<ipython-input-25-5d4f8ace93e0>:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
2000,3.504300
4000,3.468700
6000,3.454700


train/epoch,▁▄▇█
train/global_step,▁▄▇█
train/grad_norm,█▁▁
train/learning_rate,█▄▁
train/loss,█▃▁
total_flos,1.325699445030912e+17
train/epoch,2.99886
train/global_step,6561
train/grad_norm,1.0722
train/learning_rate,0.0
train/loss,3.4547


<ipython-input-25-5d4f8ace93e0>:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
2000,3.430800


In [ ]:
model.save_pretrained('drive/MyDrive/T5_fine_tuned_result/model_weights/')

In [ ]:
tokenizer.save_pretrained("drive/MyDrive/T5_fine_tuned_result/tokenizer_weights")

In [ ]:
import torch
import gc

def free_memory():
    torch.cuda.empty_cache()
    gc.collect()
free_memory()

In [ ]:
bleu_test_subt_after = bleu_score(model, tokenizer, bleu_loader, device)
print(bleu_test_subt_after)

In [ ]:
import kagglehub
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler

# Download latest version
path = kagglehub.dataset_download("averkij/russian-chinese-parallel-corpora")

print("Path to dataset files:", path)